In [ ]:
!pip install playwright pandas openpyxl
!playwright install chromium

In [ ]:
import asyncio
import os
import re
import sys
import urllib.parse
from datetime import datetime

import pandas as pd
from playwright.sync_api import sync_playwright

# KONFIGURASI
VIDEO_URL = "https://www.tiktok.com/@username/video/1234567890.."

# Jumlah komentar target.
# Bisa lebih dari 1000, misalnya 2000, 5000, dst.
MAX_COMMENTS = 1005

# Pilih format export: "csv" atau "xlsx"
EXPORT_FORMAT = "csv"

HEADLESS = False
USE_PERSISTENT_PROFILE = True
USER_DATA_DIR = "./tiktok_profile"
OUTPUT_DIR = "."

USERNAME_SOURCE = "commenter"
DATE_FORMAT = "%Y-%m-%d"
AUTO_SUFFIX_IF_EXISTS = True

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

VIEWPORT = {
    "width": 1366,
    "height": 768
}


# FUNGSI BANTU
def extract_video_id(url: str) -> str:
    match = re.search(r"/video/(\d+)", url)
    if match:
        return match.group(1)

    parsed = urllib.parse.urlparse(url)
    return parsed.path.rstrip("/").split("/")[-1]


def extract_username_from_url(url: str) -> str:
    match = re.search(r"@([^/]+)/video", url)
    if match:
        return "@" + match.group(1)

    return "N/A"


def format_username(raw) -> str:
    raw = str(raw or "").strip()

    if not raw or raw.upper() == "N/A":
        return "N/A"

    raw = raw.lstrip("@")
    return "@" + raw


def sanitize_filename(value: str, fallback: str = "tiktok") -> str:
    value = str(value or "").strip()
    value = value.lstrip("@")
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    value = re.sub(r"_{2,}", "_", value).strip("._-")

    reserved_names = {
        "CON", "PRN", "AUX", "NUL",
        "COM1", "COM2", "COM3", "COM4", "COM5", "COM6", "COM7", "COM8", "COM9",
        "LPT1", "LPT2", "LPT3", "LPT4", "LPT5", "LPT6", "LPT7", "LPT8", "LPT9"
    }

    if value.upper() in reserved_names:
        value = value + "_"

    if not value:
        value = fallback

    return value[:80]


def get_first(mapping, keys, default=None):
    if not isinstance(mapping, dict):
        return default

    for key in keys:
        if key in mapping:
            value = mapping[key]
            if value is not None and value != "":
                return value

    return default


def iter_comment_objects(payload):
    if isinstance(payload, list):
        for item in payload:
            yield from iter_comment_objects(item)

    elif isinstance(payload, dict):
        has_text = any(
            key in payload
            for key in ("text", "content", "comment", "comment_text")
        )

        has_user_or_id = any(
            key in payload
            for key in ("user", "UserInfo", "user_info", "uid", "id", "cid")
        )

        if has_text and has_user_or_id:
            yield payload

        for key in (
            "comments",
            "comment_list",
            "reply_comments",
            "reply_comment",
            "data",
            "items"
        ):
            if key in payload:
                yield from iter_comment_objects(payload[key])


def scroll_comments_js() -> str:
    return """
    () => {
      const selectors = [
        '[data-e2e="comment-list"]',
        '[data-e2e="comment-panel"]',
        '.comment-mainContent',
        '[role="dialog"]'
      ];

      for (const sel of selectors) {
        const el = document.querySelector(sel);
        if (!el) continue;

        let node = el;

        while (node && node !== document.body) {
          const style = window.getComputedStyle(node);
          const overflowY = style.overflowY || style.getPropertyValue('overflow-y');

          if (
            (overflowY === 'auto' || overflowY === 'scroll') &&
            node.scrollHeight > node.clientHeight + 10
          ) {
            node.scrollTop = node.scrollHeight;
            return true;
          }

          node = node.parentElement;
        }
      }

      window.scrollBy(0, 1200);
      return false;
    }
    """


def unique_filepath(output_dir: str, base_name: str, extension: str, auto_suffix: bool = True) -> str:
    os.makedirs(output_dir, exist_ok=True)

    candidate = os.path.join(output_dir, f"{base_name}.{extension}")

    if not auto_suffix or not os.path.exists(candidate):
        return os.path.abspath(candidate)

    counter = 1

    while True:
        candidate = os.path.join(output_dir, f"{base_name}_{counter}.{extension}")

        if not os.path.exists(candidate):
            return os.path.abspath(candidate)

        counter += 1


# SCRAPER UTAMA
def scrape_tiktok_comments_sync(
    video_url,
    max_comments=1000,
    export_format="csv",
    headless=False,
    use_persistent_profile=True,
    user_data_dir="./tiktok_profile",
    output_dir=".",
    username_source="commenter",
    date_format="%Y-%m-%d",
    auto_suffix_if_exists=True
):


    if sys.platform == "win32":
        try:
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except Exception:
            pass

    video_id = extract_video_id(video_url)
    video_owner_username = extract_username_from_url(video_url)

    owner_for_file = sanitize_filename(video_owner_username, fallback=video_id)

    scrape_date = datetime.now().strftime(date_format)

    file_base_name = f"{owner_for_file}_{scrape_date}"

    comments_data = []
    seen_keys = set()

    print("=" * 60)
    print(f"Video ID          : {video_id}")
    print(f"Video Owner       : {video_owner_username}")
    print(f"Username Source   : {username_source}")
    print(f"Target            : {max_comments} komentar")
    print(f"Export            : {str(export_format).upper()}")
    print(f"Headless          : {headless}")
    print(f"Persistent Profile: {use_persistent_profile}")
    print(f"Output Folder     : {os.path.abspath(output_dir)}")
    print(f"File Base Name    : {file_base_name}")
    print("=" * 60)

    with sync_playwright() as p:
        browser = None
        context = None

        try:
            launch_args = [
                "--disable-blink-features=AutomationControlled"
            ]

            ignore_default_args = [
                "--enable-automation"
            ]

            user_data_dir_abs = os.path.abspath(user_data_dir)
            os.makedirs(user_data_dir_abs, exist_ok=True)

            if use_persistent_profile:
                context = p.chromium.launch_persistent_context(
                    user_data_dir=user_data_dir_abs,
                    headless=headless,
                    user_agent=USER_AGENT,
                    viewport=VIEWPORT,
                    args=launch_args,
                    ignore_default_args=ignore_default_args,
                )
                page = context.pages[0] if context.pages else context.new_page()
            else:
                browser = p.chromium.launch(
                    headless=headless,
                    args=launch_args,
                    ignore_default_args=ignore_default_args,
                )
                context = browser.new_context(
                    user_agent=USER_AGENT,
                    viewport=VIEWPORT,
                )
                page = context.new_page()

            page.add_init_script(
                "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"
            )

            def on_response(response):
                try:
                    if response.status != 200:
                        return

                    url = response.url

                    if "comment/list" not in url and "/comment?" not in url:
                        return

                    data = response.json()

                    for c in iter_comment_objects(data):
                        if not isinstance(c, dict):
                            continue

                        user_obj = (
                            c.get("user")
                            or c.get("UserInfo")
                            or c.get("user_info")
                            or {}
                        )

                        uid_value = get_first(
                            user_obj,
                            (
                                "uid",
                                "id",
                                "user_id",
                                "sec_uid",
                                "secUid",
                            ),
                            None
                        )

                        if uid_value is None:
                            uid_value = get_first(
                                c,
                                (
                                    "uid",
                                    "user_id",
                                    "author_id",
                                ),
                                None
                            )

                        if uid_value is None or str(uid_value).strip() == "":
                            uid = "N/A"
                        else:
                            uid = str(uid_value).strip()

                        if username_source == "video_owner":
                            username = format_username(video_owner_username)
                        else:
                            raw_username = get_first(
                                user_obj,
                                (
                                    "uniqueId",
                                    "unique_id",
                                    "username",
                                    "user_name",
                                    "screenName",
                                    "nickname",
                                ),
                                None
                            )

                            if raw_username is None:
                                raw_username = get_first(
                                    c,
                                    (
                                        "username",
                                        "user_name",
                                    ),
                                    None
                                )

                            username = format_username(raw_username)

                        text_value = get_first(
                            c,
                            (
                                "text",
                                "content",
                                "comment",
                                "comment_text",
                            ),
                            ""
                        )

                        text = str(text_value).strip()

                        cid_value = get_first(
                            c,
                            (
                                "cid",
                                "id",
                                "comment_id",
                                "commentId",
                            ),
                            None
                        )

                        cid = str(cid_value).strip() if cid_value is not None else ""

                        like_value = get_first(
                            c,
                            (
                                "digg_count",
                                "like_count",
                                "diggCount",
                                "likeCount",
                                "likes",
                            ),
                            0
                        )

                        try:
                            if isinstance(like_value, str):
                                like_value = like_value.replace(",", "").strip()

                            like_count = int(float(like_value))
                        except Exception:
                            like_count = 0

                        key = cid if cid else f"{uid}::{text}"

                        if key in seen_keys:
                            continue

                        seen_keys.add(key)

                        comments_data.append(
                            {
                                "comment_id": cid,
                                "uid": uid,
                                "username": username,
                                "komentar": text,
                                "like": like_count,
                            }
                        )

                except Exception:
                    pass

            page.on("response", on_response)

            print("Membuka halaman TikTok...")
            page.goto(video_url, wait_until="domcontentloaded", timeout=60000)
            page.wait_for_timeout(5000)
            try:
                page.wait_for_selector(
                    '[data-e2e="comment-list"], [data-e2e="comment-panel"], .comment-mainContent',
                    timeout=20000
                )
            except Exception:
                pass

            try:
                comment_icon = page.query_selector('[data-e2e="comment-icon"]')
                if comment_icon:
                    comment_icon.click(timeout=3000)
                    page.wait_for_timeout(2000)
            except Exception:
                pass

            if len(comments_data) == 0:
                print("-" * 60)
                print("Belum ada komentar yang tertangkap.")
                print("Jika muncul CAPTCHA, verifikasi, atau minta login:")
                print("1. Selesaikan manual di browser yang terbuka")
                print("2. Script akan menunggu 60 detik")
                print("-" * 60)
                page.wait_for_timeout(60000)

            prev_count = 0
            no_new_comment_streak = 0

            print("Mulai scrolling untuk memuat komentar...")

            while len(comments_data) < max_comments:
                page.evaluate(scroll_comments_js())

                page.wait_for_timeout(2000)

                if len(comments_data) == prev_count:
                    no_new_comment_streak += 1
                    if no_new_comment_streak >= 10:
                        print("Tidak ada komentar baru. Berhenti scrolling.")
                        break
                else:
                    no_new_comment_streak = 0

                prev_count = len(comments_data)
                print(f"Komentar terkumpul: {len(comments_data)}")

        finally:
            if context is not None:
                try:
                    context.close()
                except Exception:
                    pass

            if browser is not None:
                try:
                    browser.close()
                except Exception:
                    pass

    if not comments_data:
        print("Tidak ada komentar yang berhasil dikumpulkan.")
        return None, None

    df = pd.DataFrame(comments_data)

    if "comment_id" in df.columns:
        cid_str = df["comment_id"].astype(str).str.strip()

        empty_mask = (
            cid_str.eq("")
            | cid_str.str.lower().isin(["none", "nan", "null"])
        )

        if empty_mask.any():
            df.loc[empty_mask, "comment_id"] = (
                "no_cid_" + df.index[empty_mask].astype(str)
            )

        df = df.drop_duplicates(subset=["comment_id"], keep="first")
    else:
        df = df.drop_duplicates()

    df = df.drop(columns=["comment_id"], errors="ignore")

    for col in ["uid", "username", "komentar", "like"]:
        if col not in df.columns:
            if col == "like":
                df[col] = 0
            else:
                df[col] = "N/A"

    df = df.head(max_comments)

    df = df[["uid", "username", "komentar", "like"]]

    fmt = str(export_format).lower().strip()

    if fmt == "csv":
        extension = "csv"
    elif fmt in ["xlsx", "excel", "xls"]:
        extension = "xlsx"
    else:
        raise ValueError("Export format tidak valid. Gunakan 'csv' atau 'xlsx'.")

    filepath = unique_filepath(
        output_dir=output_dir,
        base_name=file_base_name,
        extension=extension,
        auto_suffix=auto_suffix_if_exists
    )

    if extension == "csv":
        df.to_csv(filepath, index=False, encoding="utf-8-sig")
    else:
        df.to_excel(filepath, index=False)

    print("=" * 60)
    print(f"Selesai. Total baris: {len(df)}")
    print(f"File tersimpan di: {filepath}")
    print("=" * 60)

    return df, filepath


# WRAPPER UNTUK JUPYTER/VS CODE
async def scrape_tiktok_comments(
    video_url,
    max_comments=1000,
    export_format="csv",
    headless=False,
    use_persistent_profile=True,
    user_data_dir="./tiktok_profile",
    output_dir=".",
    username_source="commenter",
    date_format="%Y-%m-%d",
    auto_suffix_if_exists=True
):

    return await asyncio.to_thread(
        scrape_tiktok_comments_sync,
        video_url,
        max_comments,
        export_format,
        headless,
        use_persistent_profile,
        user_data_dir,
        output_dir,
        username_source,
        date_format,
        auto_suffix_if_exists
    )


# EKSEKUSI
df, file_path = await scrape_tiktok_comments(
    video_url=VIDEO_URL,
    max_comments=MAX_COMMENTS,
    export_format=EXPORT_FORMAT,
    headless=HEADLESS,
    use_persistent_profile=USE_PERSISTENT_PROFILE,
    user_data_dir=USER_DATA_DIR,
    output_dir=OUTPUT_DIR,
    username_source=USERNAME_SOURCE,
    date_format=DATE_FORMAT,
    auto_suffix_if_exists=AUTO_SUFFIX_IF_EXISTS
)

if df is not None:
    try:
        display(df.head(10))
    except NameError:
        print(df.head(10).to_string(index=False))

    print("\nFile hasil:")
    print(file_path)